# DGraphFin Training + Pruning Debug Notebook

Use this notebook to run the DGL-based training/evaluation pipeline for baseline or pruned GraphSAGE variants, capture intermediate metrics, and visualize aggregated results/embeddings without leaving Jupyter.


In [1]:

from __future__ import annotations

import json
import os
from dataclasses import asdict
from pathlib import Path
from typing import Dict, List, Optional

import dgl
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent

os.chdir(project_root)
print(f"Project root set to: {project_root}")

from experiments.aggregate_results import aggregate_variant, load_seed_summaries
from src.data.dgraph_fin import load_dgraphfin_dataset
from src.metrics.efficiency import count_parameters, estimate_graphsage_flops
from src.metrics.metrics import compute_binary_metrics
from src.models.graphsage import build_graphsage_for_data
from src.pruning.magnitude import prune_model_magnitude
from src.pruning.synflow import enforce_masks, prune_model_synflow
from src.training.run_experiment import (
    ExperimentConfig,
    TrainHyperparams,
    load_experiment_config,
    run_single_seed,
    save_summary,
)
from src.training.train_baseline import TrainConfig as BaseTrainConfig
from src.training.train_baseline import build_loaders, compute_pos_weight, set_seed

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True



Project root set to: /Users/jameshall/UCLA/260D/project


/Users/jameshall/UCLA/260D/project/venv/lib/python3.10/site-packages/torch_geometric/__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /Users/jameshall/UCLA/260D/project/venv/lib/python3.10/site-packages/torch_scatter/_scatter_cpu.so
  import torch_geometric.typing


In [2]:
CONFIG_CHOICES = {
    "baseline": project_root / "src" / "config" / "baseline.yaml",
    "pruned_magnitude": project_root / "src" / "config" / "pruned_magnitude.yaml",
    "pruned_synflow": project_root / "src" / "config" / "pruned_synflow.yaml",
}

for name, path in CONFIG_CHOICES.items():
    status = "OK" if path.exists() else "MISSING"
    print(f"{name:<18} -> {path} [{status}]")


baseline           -> /Users/jameshall/UCLA/260D/project/src/config/baseline.yaml [OK]
pruned_magnitude   -> /Users/jameshall/UCLA/260D/project/src/config/pruned_magnitude.yaml [OK]
pruned_synflow     -> /Users/jameshall/UCLA/260D/project/src/config/pruned_synflow.yaml [OK]


In [3]:
selected_config_key = "baseline"  # change to 'pruned_magnitude' or 'pruned_synflow'
SELECTED_CONFIG_PATH = CONFIG_CHOICES[selected_config_key]
SELECTED_CONFIG_PATH


PosixPath('/Users/jameshall/UCLA/260D/project/src/config/baseline.yaml')

In [4]:
exp_cfg = load_experiment_config(str(SELECTED_CONFIG_PATH))
print(exp_cfg)
print(json.dumps(asdict(exp_cfg), indent=2, default=str))


ExperimentConfig(data_root='data/DGraphFin2', output_dir='experiments/results/baseline', model_variant='baseline', sampling='uniform', sparsity=0.0, train=TrainHyperparams(epochs=20, lr=0.001, weight_decay=5e-05, hidden_channels=128, num_layers=2, dropout=0.2, batch_size=2048, num_neighbors=[25, 25], num_hops=2, k_pos=25, k_neg=5, device='mps'), seeds=[1, 2, 3, 4, 5])
{
  "data_root": "data/DGraphFin2",
  "output_dir": "experiments/results/baseline",
  "model_variant": "baseline",
  "sampling": "uniform",
  "sparsity": 0.0,
  "train": {
    "epochs": 20,
    "lr": 0.001,
    "weight_decay": 5e-05,
    "hidden_channels": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "batch_size": 2048,
    "num_neighbors": [
      25,
      25
    ],
    "num_hops": 2,
    "k_pos": 25,
    "k_neg": 5,
    "device": "mps"
  },
  "seeds": [
    1,
    2,
    3,
    4,
    5
  ]
}


In [5]:

def build_base_train_config(exp_cfg: ExperimentConfig, seed: int) -> BaseTrainConfig:
    t = exp_cfg.train
    return BaseTrainConfig(
        data_root=exp_cfg.data_root,
        epochs=t.epochs,
        lr=t.lr,
        weight_decay=t.weight_decay,
        hidden_channels=t.hidden_channels,
        num_layers=t.num_layers,
        dropout=t.dropout,
        batch_size=t.batch_size,
        num_neighbors=t.num_neighbors,
        num_hops=t.num_hops,
        sampling=exp_cfg.sampling,
        k_pos=t.k_pos,
        k_neg=t.k_neg,
        device=t.device,
        seed=seed,
    )


def run_single_seed_with_capture(exp_cfg: ExperimentConfig, seed: int) -> Dict[str, object]:
    base_cfg = build_base_train_config(exp_cfg, seed)
    set_seed(base_cfg.seed)
    device = torch.device(base_cfg.device)

    dataset = load_dgraphfin_dataset(root=exp_cfg.data_root)
    graph_cpu = dataset[0]
    graph_device = graph_cpu.to(device)

    model = build_graphsage_for_data(
        graph_device,
        hidden_channels=base_cfg.hidden_channels,
        num_layers=base_cfg.num_layers,
        dropout=base_cfg.dropout,
    ).to(device)

    if exp_cfg.model_variant == "pruned_magnitude":
        model = prune_model_magnitude(model, sparsity=exp_cfg.sparsity)
    elif exp_cfg.model_variant == "pruned_synflow":
        model = prune_model_synflow(model, graph=graph_device, sparsity=exp_cfg.sparsity)
    elif exp_cfg.model_variant != "baseline":
        raise ValueError(f"Unknown model_variant: {exp_cfg.model_variant}")

    num_params = count_parameters(model, trainable_only=True)
    approx_flops = estimate_graphsage_flops(
        model, num_nodes=graph_cpu.num_nodes(), num_edges=graph_cpu.num_edges()
    )

    labels = graph_device.ndata["label"]
    pos_weight = compute_pos_weight(labels).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=base_cfg.lr, weight_decay=base_cfg.weight_decay)
    train_loader = build_loaders(graph_cpu, base_cfg, device)

    history: List[Dict[str, float]] = []
    best_val_auprc = float("-inf")
    best_epoch = -1
    best_val_metrics: Dict[str, float] = {}
    best_test_metrics: Dict[str, float] = {}

    for epoch in range(1, base_cfg.epochs + 1):
        model.train()
        total_loss = 0.0
        total_examples = 0

        if base_cfg.sampling == "uniform":
            for _, _, blocks in train_loader:
                optimizer.zero_grad()
                blocks = [block.to(device) for block in blocks]
                batch_feats = blocks[0].srcdata["feat"]
                batch_labels = blocks[-1].dstdata["label"].float()
                logits = model(blocks, batch_feats).view(-1)
                loss = criterion(logits, batch_labels)
                loss.backward()
                optimizer.step()

                if exp_cfg.model_variant in ("pruned_magnitude", "pruned_synflow"):
                    enforce_masks(model)

                batch_size = batch_labels.size(0)
                total_loss += loss.item() * batch_size
                total_examples += batch_size
        else:
            for batch in train_loader:
                optimizer.zero_grad()
                subgraph = batch.graph.to(device)
                feats = subgraph.ndata["feat"]
                logits_all = model(subgraph, feats).view(-1)
                seed_idx = batch.seed_idx.to(logits_all.device)
                targets = subgraph.ndata["label"][seed_idx].float()
                logits = logits_all[seed_idx]
                loss = criterion(logits, targets)
                loss.backward()
                optimizer.step()

                if exp_cfg.model_variant in ("pruned_magnitude", "pruned_synflow"):
                    enforce_masks(model)

                batch_size = targets.size(0)
                total_loss += loss.item() * batch_size
                total_examples += batch_size

        avg_loss = total_loss / max(total_examples, 1)

        model.eval()
        with torch.no_grad():
            logits_full = model(graph_device, graph_device.ndata["feat"]).view(-1)

            def _eval_mask(mask: torch.Tensor) -> Dict[str, float]:
                mask_logits = logits_full[mask]
                mask_targets = graph_device.ndata["label"][mask].float()
                loss_val = criterion(mask_logits, mask_targets).item()
                probs_val = torch.sigmoid(mask_logits)
                metrics = compute_binary_metrics(mask_targets.cpu(), probs_val.cpu())
                metrics["loss"] = loss_val
                return metrics

            val_metrics = _eval_mask(graph_device.ndata["val_mask"])
            test_metrics = _eval_mask(graph_device.ndata["test_mask"])

        history.append(
            {
                "epoch": epoch,
                "train_loss": avg_loss,
                "val_loss": val_metrics["loss"],
                "val_auprc": val_metrics["auprc"],
                "val_roc_auc": val_metrics["roc_auc"],
            }
        )

        if val_metrics["auprc"] > best_val_auprc:
            best_val_auprc = val_metrics["auprc"]
            best_epoch = epoch
            best_val_metrics = val_metrics
            best_test_metrics = test_metrics

    model.eval()
    with torch.no_grad():
        logits_full = model(graph_device, graph_device.ndata["feat"]).view(-1)
        probs_full = torch.sigmoid(logits_full).detach().cpu()
        hidden = graph_device.ndata["feat"]
        for conv in model.convs:
            hidden = conv(graph_device, hidden)
            hidden = model.activation(hidden)
            hidden = model.dropout(hidden)
        embeddings = hidden.detach().cpu()

    summary = {
        "seed": seed,
        "model_variant": exp_cfg.model_variant,
        "sampling": exp_cfg.sampling,
        "sparsity": exp_cfg.sparsity,
        "num_params": num_params,
        "approx_flops": approx_flops,
        "best_epoch": best_epoch,
        "best_val": best_val_metrics,
        "best_test": best_test_metrics,
        "final_val_auprc": float(history[-1]["val_auprc"]),
        "final_test_auprc": float(best_test_metrics.get("auprc", float("nan"))),
    }

    capture = {
        "summary": summary,
        "history": history,
        "logits": logits_full.detach().cpu(),
        "probabilities": probs_full,
        "embeddings": embeddings,
        "labels": graph_device.ndata["label"].detach().cpu(),
        "masks": {
            "train": graph_device.ndata["train_mask"].detach().cpu(),
            "val": graph_device.ndata["val_mask"].detach().cpu(),
            "test": graph_device.ndata["test_mask"].detach().cpu(),
        },
        "model_state_dict": model.cpu().state_dict(),
        "seed": seed,
        "config": asdict(exp_cfg),
    }

    return capture



In [8]:

import dgl
print(f"DGL version: {dgl.__version__}")



ModuleNotFoundError: No module named 'torch_sparse'

In [6]:

def execute_notebook_run(
    config_path: Path,
    seeds: Optional[List[int]] = None,
    capture_model: bool = True,
    persist_summaries: bool = True,
):
    exp_cfg = load_experiment_config(str(config_path))
    seeds_to_run = seeds or exp_cfg.seeds
    captures = []

    for seed in seeds_to_run:
        print(f"=== Running {exp_cfg.model_variant} | seed={seed} ===")
        if capture_model:
            capture = run_single_seed_with_capture(exp_cfg, seed)
            summary = capture["summary"]
        else:
            summary = run_single_seed(exp_cfg, seed)
            capture = {"summary": summary, "history": []}
        if persist_summaries:
            save_summary(exp_cfg.output_dir, seed, summary)
        captures.append(capture)
    return exp_cfg, captures



=== Running baseline | seed=42 ===


ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'

In [ ]:

# Example (commented to avoid surprise training run):
# exp_cfg, captures = execute_notebook_run(SELECTED_CONFIG_PATH, seeds=[42], capture_model=True)
# captures[-1]["summary"]



=== Running baseline | seed=42 ===


ImportError: 'NeighborSampler' requires either 'pyg-lib' or 'torch-sparse'

In [ ]:
def plot_training_history(history: List[Dict[str, float]]):
    if not history:
        print("No history captured. Re-run with capture_model=True.")
        return
    epochs = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss = [h["val_loss"] for h in history]
    val_auprc = [h["val_auprc"] for h in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, train_loss, label="train_loss")
    axes[0].plot(epochs, val_loss, label="val_loss")
    axes[0].set_title("Loss curves")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, val_auprc, label="val_auprc", color="#C44E52")
    axes[1].set_title("Validation AUPRC")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("AUPRC")
    axes[1].set_ylim(0.0, 1.0)
    plt.tight_layout()
    plt.show()


# Example usage after a run:
plot_training_history(captures[-1]["history"])


In [13]:
def pca_project(tensor: torch.Tensor, n_components: int = 2) -> np.ndarray:
    tensor = torch.as_tensor(tensor, dtype=torch.float32)
    tensor = tensor - tensor.mean(dim=0, keepdim=True)
    q = min(n_components + 2, tensor.size(1))
    U, S, V = torch.pca_lowrank(tensor, q=q)
    coords = tensor @ V[:, :n_components]
    return coords.cpu().numpy()


def plot_embedding_projection(
    embeddings: torch.Tensor,
    labels: torch.Tensor,
    probs: Optional[torch.Tensor] = None,
    title: str = "Node embeddings (PCA)",
):
    coords = pca_project(embeddings, n_components=2)
    labels_np = labels.cpu().numpy()
    plt.figure(figsize=(6, 5))
    scatter = plt.scatter(
        coords[:, 0],
        coords[:, 1],
        c=labels_np,
        cmap="coolwarm",
        s=5,
        alpha=0.6,
    )
    plt.title(title + " | color = label")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.colorbar(scatter, label="Label")
    plt.show()

    if probs is not None:
        probs_np = probs.cpu().numpy()
        plt.figure(figsize=(6, 5))
        scatter = plt.scatter(
            coords[:, 0],
            coords[:, 1],
            c=probs_np,
            cmap="viridis",
            s=5,
            alpha=0.6,
        )
        plt.title(title + " | color = predicted fraud prob")
        plt.xlabel("PC1")
        plt.ylabel("PC2")
        plt.colorbar(scatter, label="P(fraud)")
        plt.show()


# After a run:
plot_embedding_projection(captures[-1]["embeddings"], captures[-1]["labels"], captures[-1]["probabilities"])


NameError: name 'captures' is not defined

In [15]:
RESULTS_ROOT = project_root / "experiments" / "results"


def aggregate_results_table(results_root: Path = RESULTS_ROOT) -> List[Dict[str, object]]:
    rows: List[Dict[str, object]] = []
    for variant_dir in sorted(results_root.glob("*")):
        if not variant_dir.is_dir():
            continue
        summaries = load_seed_summaries(variant_dir)
        if not summaries:
            continue
        agg = aggregate_variant(summaries)
        agg["results_dir"] = str(variant_dir)
        rows.append(agg)
    return rows


def display_results_table(rows: List[Dict[str, object]]):
    if not rows:
        print(f"No result JSON files detected under {RESULTS_ROOT}. Run a training cell first.")
        return
    header = (
        "Variant",
        "Sampling",
        "Sparsity",
        "Best Test AUPRC",
        "ROC-AUC",
        "F1",
        "#Params",
        "FLOPs",
        "#Seeds",
    )
    print(" | ".join(header))
    print(" | ".join(["---"] * len(header)))
    for row in rows:
        if row["approx_flops"] is None:
            flops_str = "N/A"
        else:
            flops_str = f"{row['approx_flops']:.3e}"
        print(
            f"{row['model_variant']} | {row['sampling']} | {row['sparsity']:.2f} | "
            f"{row['best_test_auprc_mean']:.4f}±{row['best_test_auprc_std']:.4f} | "
            f"{row['best_test_roc_auc_mean']:.4f}±{row['best_test_roc_auc_std']:.4f} | "
            f"{row['best_test_f1_mean']:.4f}±{row['best_test_f1_std']:.4f} | "
            f"{row['num_params']} | {flops_str} | "
            f"{row['num_seeds']}"
        )


# rows = aggregate_results_table()
display_results_table(rows)


NameError: name 'rows' is not defined

In [16]:
def plot_aggregated_rows(rows: List[Dict[str, object]]):
    if not rows:
        print("No aggregated rows to plot.")
        return
    variants = [r["model_variant"] for r in rows]
    auprc_means = [r["best_test_auprc_mean"] for r in rows]
    auprc_stds = [r["best_test_auprc_std"] for r in rows]
    flops = [r["approx_flops"] for r in rows]

    plt.figure(figsize=(6, 4))
    x = np.arange(len(variants))
    plt.bar(x, auprc_means, yerr=auprc_stds, capsize=4)
    plt.xticks(x, variants, rotation=30)
    plt.ylabel("Best Test AUPRC")
    plt.title("Best Test AUPRC by Variant")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(6, 4))
    for v, a, f in zip(variants, auprc_means, flops):
        if f is None:
            continue
        plt.scatter(f, a, label=v)
        plt.text(f, a, v)
    plt.xscale("log")
    plt.xlabel("Approx FLOPs (log scale)")
    plt.ylabel("Best Test AUPRC")
    plt.title("AUPRC vs FLOPs")
    plt.tight_layout()
    plt.show()


# rows = aggregate_results_table()
plot_aggregated_rows(rows)


NameError: name 'rows' is not defined

In [12]:
def find_latest_summary(results_root: Path = RESULTS_ROOT) -> Optional[Path]:
    json_files = sorted(results_root.glob("**/seed_*.json"), key=lambda p: p.stat().st_mtime, reverse=True)
    return json_files[0] if json_files else None


def load_summary(path: Path) -> Dict[str, object]:
    with open(path, "r") as f:
        return json.load(f)


latest = find_latest_summary()
if latest:
    print(f"Latest summary: {latest}")
    display_json = load_summary(latest)
    print(json.dumps(display_json, indent=2))
else:
    print("No seed summaries saved yet.")


No seed summaries saved yet.


## Recommended workflow

1. Pick a config via `selected_config_key` and inspect the parsed dataclass.
2. Uncomment `execute_notebook_run(...)` to launch a seed (set `capture_model=True` to enable visualizations).
3. Call `plot_training_history(...)` and `plot_embedding_projection(...)` on the returned capture to inspect dynamics.
4. Use `aggregate_results_table()` followed by `plot_aggregated_rows()` to compare variants after multiple runs.
5. Re-run the "Latest summary" cell at the bottom anytime you need a quick JSON snapshot for reporting.
